# AIFM Hedge Fund

AIFMD follows a **principles-based** approach to risk management.

**Appropriate · Adequate · Effective · Independent · Consistent · Documented**

Rather than prescribing a specific market risk methodology, confidence level or risk limit framework, AIFMD requires the AIFM to (1) establish, (2) maintain and (3) document **risk measurement** and **monitoring processes** appropriate to the fund's:
- investment strategy
- risk profile
- portfolio composition

The risk framework, methodologies, limits and escalation procedures are typically defined within the Risk Management Policy (RMP) and supporting governance documentation, and reflected in regulatory disclosures and reporting.

---

### Fund Example in this Notebook

* AIFM Hedge Fund
* Strategy: Long/short equity, bonds, FX and options
* Domicile: Luxembourg

The notebook uses a representative hedge fund profile to illustrate selected AIFMD risk management concepts. Fund characteristics, risk limits, methodologies and reporting parameters are simulated but are intended to reflect the type of framework that would typically be defined within a fund's Risk Management Policy (RMP) and supporting governance documentation.

Consistent with the principles-based approach of AIFMD, the risk measures, limits and monitoring processes shown in this notebook should be viewed as examples of an appropriately documented risk framework rather than prescribed regulatory requirements.

---

### Implementation Context

The fund operates on a simulated operating model consisting of portfolio positions, reference data, counterparties, fund characteristics and market data stored in a SQLite database. Market data is sourced through a simulated Bloomberg workflow and consumed by the risk analytics layer for risk measurement, stress testing and regulatory reporting.

The notebooks focus on analysis and interpretation, while data aggregation, calculations and reporting logic are implemented within the underlying Python package.

---

### Regulatory Framework

The regulatory framework referenced in this notebook includes AIFMD legislation, delegated regulations, ESMA guidance and Luxembourg implementation measures relevant to a Luxembourg-domiciled AIFM.

<small>

| | AIFMD I | AIFMD II / UCITS VI |
|---|---|---|
| **Directive** | 2011/61/EU | 2024/927 |
| **Luxembourg law** | Law of 12 July 2013 | Law of 3 March 2026 |
| **EU transposition deadline** | 22 July 2013 | 16 April 2026 |

NOTE: Directive (EU) 2024/927 amends AIFMD and UCITS, including liquidity management tools, loan origination, delegation and supervisory reporting changes.
</small>

### Binding Technical Standards

<small>

| Reference | Scope |
|---|---|
| Delegated Regulation 231/2013 | Risk management, leverage (Art. 6-11), liquidity management and Annex IV reporting |
| Delegated Regulation 2026/466 | Liquidity Management Tools (LMTs)|

Note: The Regulatory Technical Standards (RTS) are drafted by ESMA and subsequently adopted by the European Commission as binding Commission Delegated Regulations.

</small>

### ESMA Guidance

<small>

| Reference | Scope |
|---|---|
| ESMA AIFMD Reporting Technical Guidance v1.7 | Annex IV reporting framework|
| ESMA LST Guidelines (ESMA/2020/1498) | Liquidity stress testing framework |
| ESMA LMT Guidelines (ESMA34-671404336-1364) | Selection and calibration of LMTs |

Note: ESMA Guidelines (GL) are subject to a comply-or-explain framework at the level of national supervisory authorities. Where adopted, they become part of the supervisory expectations applicable AIFMs and UCITS management companies. ESMA technical guidance, while not formal legislation, is also widely used to support consistent regulatory implementation and reporting practices.

</small>






In [ ]:
from src.config import VALUATION_DATE
from src.risk.reg_constants import CONFIDENCE, HORIZON, NOTICE

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.setup_db import run as setup_db
setup_db()

from src.data.database import get_engine, query_positions
from src.data.enrichment import get_risk_ready_df
from src.data.mock_bloomberg import MockBloomberg as Bloomberg
from src.risk.leverage_config import INSTRUMENT_SOURCE
import src.risk.risk_utils as rk
import src.ui.print_html_utils as phtml



FUND_ID = 'AIFM_HedgeFund'
ENGINE = get_engine()
BBG = Bloomberg()


In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
phtml.display_fund_overview_banner(fund_id=FUND_ID,engine=ENGINE)

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
)

In [ ]:
# query from SQLdb
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)

# enrich w/ info from Bloomberg
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# display the enriched risk dataframe first 2 rows
risk_df.head(2)

From this point onward, the notebook uses helper functions to render HTML views and summary tables. The aggregation, calculation and reporting logic is kept inside the package modules rather than inside the notebook, so the notebook remains focused on interpretation and review.

In [ ]:
# info in risk_df covers many risk inputs
NAV = risk_df['market_value_eur'].sum()

phtml.display_fund_summary(FUND_ID, VALUATION_DATE, positions, risk_df, NAV, valuation_date=VALUATION_DATE)


In [ ]:
phtml.display_top_positions(risk_df, valuation_date=VALUATION_DATE)


In [ ]:
# Asset class breakdown
phtml.display_asset_class_breakdown(risk_df, valuation_date=VALUATION_DATE)  # Also computes NAV internally


In [ ]:
from src.ui.plot_utils import plot_breakdown_horizontal

plot_breakdown_horizontal(
    risk_df,
    title='Asset Class Breakdown',
    group_by='asset_class',
    fund_id=FUND_ID,
    filename="01. Asset Breakdown HF",
    valuation_date=VALUATION_DATE
)


## 2. VaR and Expected Shortfall

Consistent with AIFMD's principles-based approach to risk management, the fund has selected and documented a market risk framework appropriate to its investment strategy, risk profile and investment objectives. This includes the choice of risk measures, methodologies and associated parameters used for risk measurement.

Market risk for this fund is monitored using Historical VaR and Expected Shortfall.

* 99% confidence level
* 20-day holding period (time horizon over which potential losses are measured)
* 250-day lookback period (historical observation period used to estimate the return distribution)
* Historical simulation

> Note 1: The choice of VaR methodology, confidence level (99%), holding period (20 days), and lookback window (250 days) is documented in the fund's Risk Management Policy as appropriate to its investment strategy, portfolio composition, and risk profile. AIFMD requires documented, appropriate and consistent risk measurement; it does not prescribe specific numerical parameters.

> Note 2: Historical simulation, parametric and Monte Carlo approaches are commonly used for VaR estimation. The methodology adopted depends on the fund's investment strategy, portfolio characteristics and risk management framework.

> **Limitation**: In this  VaR implementation, bond risk is approximated using duration-based sensitivity to historical yield movements. Credit spread risk, issuer-specific events and rating migration effects are not explicitly modelled.

In [ ]:
# 1-day fixed-position VaR (99% — everyday limit)
from src.pipeline.fixed_position_var import compute_fixed_position_var_1day

var_result = compute_fixed_position_var_1day(
    engine=ENGINE,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE,
    confidence=CONFIDENCE,  # 99%
)

nav = var_result['nav_eur']

# Historical VaR/ES
var_1d = var_result['var_pct']
es_1d = var_result['es_pct']
var_20d = var_result['var_scaled_pct']
es_20d = var_result['es_scaled_pct']

# Parametric VaR/ES
var_param_1d = var_result['var_param_pct']
es_param_1d = var_result['es_param_pct']
var_param_20d = var_result['var_param_scaled_pct']
es_param_20d = var_result['es_param_scaled_pct']


# Then call with the model parameter
phtml.display_var_es(var_1d, var_20d, es_1d, es_20d, nav, 
                     model='Historical', valuation_date=VALUATION_DATE)
phtml.display_var_es(var_param_1d, var_param_20d, es_param_1d, es_param_20d, nav, 
                     model='Parametric', valuation_date=VALUATION_DATE)


### 2.2 Liquidity-Adjusted VaR

Standard VaR assumes positions can be liquidated instantly at market price.
LVaR extends this by adding the estimated cost of unwinding positions under
stressed market conditions:

$$\text{LVaR} = \text{VaR} + \text{Liquidity Cost}$$

$$\text{Liquidity Cost}_i = \frac{1}{2} \times \text{spread}_i \times \text{stress multiplier}_i \times |MV_i|$$

Liquidity-Adjusted VaR is commonly discussed in banking risk management literature and internal risk frameworks as a way of incorporating liquidity costs into market risk measurement (Amihud & Mendelson, BIS working papers). It is used internally by risk managers to capture the liquidity dimension of market risk.

Spreads and stress multipliers are illustrative values adopted in this
notebook. In practice these are internal RMP parameters calibrated by the
fund manager and reviewed periodically.

<small> 

| Asset Class      | Normal Spread | Stress Multiplier | Stressed Spread       |
|------------------|---------------|-------------------|-----------------------|
| Large cap equity | 5bps          | 3x                | 15bps                 |
| Small cap equity | 20bps         | 5x                | 100bps                |
| IG bonds         | 10bps         | 5x                | 50bps                 |
| HY bonds         | 50bps         | 10x               | 500bps                |
| Senior loans     | 100bps        | 20x               | 2000bps               |
| Listed REITs     | 15bps         | 5x                | 75bps                 |
| Direct property  | N/A           | N/A               | 5-8% transaction cost |

</small> 


> Note: While Liquidity-Adjusted VaR incorporates estimated liquidation costs into market risk measurement, AIFMD liquidity risk management extends beyond trading liquidity and includes liquidity profiling, redemption stress testing and Liquidity Management Tools (LMTs). These topics are covered in Section 5.

In [ ]:
# 2.2 Liquidity-adjusted VaR
# Standard VaR + estimated cost to liquidate under stress

positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)
lvar_result = rk.liquidity_adjusted_var(var_1d, positions, stress_multiplier=5)

phtml.display_lvar(lvar_result, nav, valuation_date=VALUATION_DATE)


## 3. VaR Backtesting and Statistical Diagnostics

Consistent with the fund's documented risk measurement framework, VaR estimates should be periodically compared against realised P&L to assess whether the model remains appropriate for the portfolio and market conditions.

The backtesting framework used in this notebook follows conventions commonly associated with UCITS VaR methodologies, including a 250-observation window and breach monitoring against realised daily P&L.

Two additional statistical diagnostics are included:

- **Kupiec POF Test**: evaluates whether the observed breach frequency is consistent with the expected breach rate (1% for a 99% VaR model).
- **Christoffersen Independence Test**: evaluates whether breaches occur independently over time rather than clustering.

> **Note 1:** AIFMD does not prescribe a specific VaR backtesting methodology. The use of a 250-observation window and breach monitoring is consistent with methodologies commonly used in UCITS VaR frameworks and frequently adopted in practice where VaR is used.

> **Note 2:** Kupiec and Christoffersen tests are widely used quantitative diagnostics but are not explicitly required by AIFMD, ESMA or CSSF guidance.

> **Private Debt Note:** NAV returns for private debt funds are typically derived from periodic valuation processes rather than continuously traded market prices. As a result, observed volatility and VaR breach frequencies may be lower than for liquid market portfolios and should be interpreted accordingly.


### Backtesting as Internal Control

The fund uses a 250-observation breach-monitoring window and statistical diagnostics (Kupiec POF, Christoffersen independence) to assess whether the VaR model remains appropriate to the current portfolio and market conditions. These diagnostics are used internally for risk governance and VaR model validation.

UCITS VaR frameworks have historically used breach-count monitoring over a 250-observation period. This approach is often illustrated using Basel traffic-light terminology:

- **Green**: low number of breaches, model generally considered acceptable.
- **Amber**: elevated breach count, review and explanation may be required.
- **Red**: breach count suggests the model may require recalibration or replacement.

In [ ]:
# VaR Backtest — rolling 1-day VaR over 250 trading days (changeable)
from src.risk.var_backtest import compute_var_backtest_rolling, create_backtest_report
import pandas as pd

# Configuration
BACKTEST_LOOKBACK = 250  # Number of business days for historical reconstruction

# Compute rolling VaR backtest
start_date = (pd.Timestamp(VALUATION_DATE) - pd.tseries.offsets.BDay(BACKTEST_LOOKBACK)).strftime('%Y-%m-%d')

backtest_df = compute_var_backtest_rolling(
    engine=ENGINE,
    fund_id=FUND_ID,
    start_date=start_date,
    end_date=VALUATION_DATE,
    lookback=BACKTEST_LOOKBACK,
)

print(f"✓ Backtest computed: {len(backtest_df)} trading days")
print(f"  Lookback period: {BACKTEST_LOOKBACK} days (per date)")
print(f"  Confidence levels: 95%, 97.5%, 99%")

# Create report with Kupiec & Christoffersen tests (95%, 97.5%, 99%)
report = create_backtest_report(backtest_df)

# Display using HTML table
phtml.display_backtest_report(report, window_size=BACKTEST_LOOKBACK, valuation_date=VALUATION_DATE, model="Historical")


In [ ]:
from src.ui.backtest_plot import plot_var_backtest

# Extract properly aligned returns and VaR for plotting
# VaR[d] is tested against return FROM d TO d+1
# So we use returns[1:] and vars[:-1]
returns_shifted = backtest_df['realised_return'].iloc[1:].values
var_shifted = backtest_df['var_99_pct'].iloc[:-1].values
dates_shifted = backtest_df['date'].iloc[1:].values

# Extract Kupiec and Christoffersen p-values from report (99% confidence)
kupiec_pvalue = report[report['confidence'] == 99]['kupiec_p'].values[0] if len(report[report['confidence'] == 99]) > 0 else None
chris_pvalue = report[report['confidence'] == 99]['christoffersen_p'].values[0] if len(report[report['confidence'] == 99]) > 0 else None

fig, ax = plot_var_backtest(
    dates_shifted,
    returns_shifted,
    var_shifted,
    FUND_ID,
    title=f'VaR Backtest — {FUND_ID} (99% confidence, 250-day lookback)',
    kupiec_pvalue=kupiec_pvalue,
    christoffersen_pvalue=chris_pvalue,
    valuation_date=VALUATION_DATE
)


---

## 4. Leverage

AIFMD requires leverage to be reported using two methods:

- **Gross method**: sum of absolute exposures divided by NAV. No netting allowed.
  Derivatives converted to equivalent underlying exposure.
- **Commitment method**: hedging and netting arrangements are recognised.
  Offsetting positions in the same underlying reduce exposure.

Limits are set in the fund's offering document and reported quarterly to the CSSF
in Annex IV. AIFMD II (Directive 2024/927/EU) expanded reporting requirements,
including the breakdown by:
* asset class
* instrument type
* source: 
  - financial borrowing
  - synthetic leverage through derivatives
  - repo/reverse repo

The expanded disclosure makes it easier for regulators to identify funds building leverage through
derivatives rather than borrowing.

In [ ]:
from src.risk.leverage_computation import compute_leverage, build_bbg_maps

BORROWINGS_EUR = 0.0

lev = compute_leverage(risk_df, NAV, BBG, FUND_ID, borrowings_eur=BORROWINGS_EUR)

# Extract maps for pre-trade checks
currency_bbg_map, deriv_bbg_map = build_bbg_maps(FUND_ID)

gross_exposure = lev['gross_exposure']
gross_leverage = lev['gross_leverage']
commitment_exposure = lev['commitment_exposure']
commitment_leverage = lev['commitment_leverage']
deriv_notional_commitment = lev['deriv_notional_commitment']

risk_df = lev['risk_df']


In [ ]:
# AIFMD II granular leverage breakdown
granular = risk_df.groupby(['asset_class', 'sub_asset_class']).agg(
    gross_eur=('gross_exposure', 'sum'),
    n_positions=('isin', 'count')
    ).reset_index()

granular['gross_x_nav'] = granular['gross_eur'] / NAV
granular['source']      = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[0], axis=1)
granular['listed_otc']  = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[1], axis=1)

# exclude rows with zero gross exposure (cash)
granular = granular[granular['gross_eur'] > 0].sort_values('gross_eur', ascending=False)

# borrowings row — included per Recital 13; appended separately as it is not
# a position in risk_df but a funding liability
if BORROWINGS_EUR > 0:
    import pandas as pd
    borrow_row = pd.DataFrame([{
        'asset_class'   : 'Borrowing',
        'sub_asset_class': 'PB Financing',
        'gross_eur'     : BORROWINGS_EUR,
        'n_positions'   : 1,
        'gross_x_nav'   : BORROWINGS_EUR / NAV,
        'source'        : 'Financial Borrowing',
        'listed_otc'    : 'N/A',
    }])
    granular = pd.concat([granular, borrow_row], ignore_index=True)

In [ ]:
phtml.display_granular(granular, NAV, valuation_date=VALUATION_DATE)


---

## 5. Market and Liquidity Stress Testing
AIFMD Article 16 and the Risk Management Policy require AIFMs to conduct regular stress tests covering market, liquidity, 
and counterparty risk. Scenarios must be documented in the RMP, reviewed at least annually, 
and results reported to the CSSF via the Annex IV filing.

Unlike UCITS, AIFMD does not prescribe specific stress scenarios. The scenarios below are 
designed to reflect the material risk factors of a long/short equity fund with fixed income, 
FX, and derivative exposure. Historical scenarios — aligned with those prescribed under UCITS 
guidelines — are included alongside hypothetical shocks as a matter of good risk management 
practice and to facilitate comparability across fund types.

In production, AIFMs typically delegate 
full revaluation stress testing to third-party risk systems  (Bloomberg PORT, Axioma, or 
BlackRock Aladdin) which capture non-linear effects such as option gamma and interest rate 
convexity. For simplicity here we use a stress methodology that applies a first-order sensitivity approximation (delta-normal). This could be reasonably used for internal monitoring purposes.

$$\Delta P_i = \text{sensitivity}_i \times \text{shock}_i \times MV_i$$

Scenarios covered:
- **Equity crash**: equity markets down 30%
- **Rate shock**: parallel shift up 200bps
- **Credit widening**: spreads widen 150bps
- **FX stress**: USD and GBP depreciate 15% vs EUR
- **Combined**: simultaneous equity, rate, credit and FX shock
- **Historical**: 2008 financial crisis, 2020 COVID crash, 2022 rate shock

> **Methodology note**: in this project, stress P&L uses first-order sensitivities (delta for
> equities, modified duration for rates and credit, direct revaluation for FX).
> In production these figures come from a third-party risk system or modeled to higher orders.

In [ ]:
from src.risk.risk_utils import HISTORICAL_SCENARIOS
phtml.display_historical_scenarios(HISTORICAL_SCENARIOS)


In [ ]:

custom_scenarios = {
    'Equity Crash -30%'      : rk.stress_equity(risk_df, delta_equity=-0.30),
    'Rate Shock +200bps'     : rk.stress_rates(risk_df, delta_y=0.02),
    'Credit Widening +150bps': rk.stress_credit(risk_df, delta_spread=0.015),
    'FX Stress USD/GBP -15%' : rk.stress_fx(risk_df, fx_shocks={'USD': -0.15, 'GBP': -0.15}),
    'Combined'               : rk.stress_combined(risk_df),
}
    
phtml.display_scenarios(risk_df, custom_scenarios, add_historical=True, valuation_date=VALUATION_DATE)


---
## 6. Liquidity Profile

### 6.1 Buckets


ESMA guidelines (ESMA34-39-897) require AIFMs to categorise portfolio assets
by time to liquidation under normal market conditions. Results are reported
to the CSSF via Annex IV and used to assess redemption coverage.

Liquidity buckets (ESMA standard):

<small>

| Bucket | Time to liquidate |
|--------|------------------|
| 1      | 1 day            |
| 2      | 2-7 days         |
| 3      | 8-30 days        |
| 4      | 31-90 days       |
| 5      | 91-365 days      |
| 6      | > 1 year         |

</small>


ESMA buckets are roughly: day, week, month, quarter, year, or above.

Days to liquidate per position:

$$\text{days}_i = \frac{|MV_i|}{ADV_i \times \text{pct\_adv}}$$

* **ADV** (Average Daily Volume): average contracts/shares traded per day over
a 20-day window, sourced from Bloomberg. 
* **pct_adv = 25%** is an internal
RMP parameter representing the maximum fraction of ADV tradeable per day
without significant market impact. 
* Direct properties and instruments with
zero ADV are classified as illiquid (> 1 year).
* Cash and money market instruments are classified as 1 day by definition,
as they require no liquidation process.

In [ ]:
# MRS-30: Liquidity profile
liq = rk.compute_liquidity_profile(risk_df, NAV, pct_adv=0.25)
risk_df_liq = liq['risk_df_liq']
bucket_full = liq['bucket_full']

phtml.display_buckets(bucket_full, risk_df_liq, NAV, valuation_date=VALUATION_DATE)


In [ ]:
from src.ui.liquidity_plot import plot_liquidity_profile
x = plot_liquidity_profile(bucket_full, FUND_ID, metric='pct_nav_abs', valuation_date=VALUATION_DATE)


---
### 6.2 Redemption Stress

AIFMD Article 16 and ESMA/2020/1498 require AIFMs to assess whether the
fund can meet redemptions under normal and stressed conditions. Assets
liquidatable within the contractual notice period (buckets `'1 day'` and
`'2-7 days'`) are compared against the redemption amount. A shortfall
triggers a gate or side-pocket recommendation to the risk committee.

<small>

| Scenario | Redemption |  Rationale |
| --- | --- | --- |
| Normal | 10% NAV | Routine investor exit |
| Large | 25% NAV | Large single redemption |
| Stress | 50% NAV | Co-ordinated stress exit |
| Largest investor | Fund-specific | Concentration stress |

In [ ]:
REDEMPTION_SCENARIOS = [
    (0.10, 'Normal'),
    (0.25, 'Large'),
    (0.50, 'Stress'),
]
phtml.display_redemption_stress(FUND_ID, NOTICE, REDEMPTION_SCENARIOS, NAV, risk_df_liq, valuation_date=VALUATION_DATE)


### 6.3 Investor Concentration

**Internal Concentration Monitoring (fund-level governance):**
- Single investor > 20% of NAV is flagged as concentration risk for escalation to PM and CRO
- Top 3 investors > 50% of NAV is flagged as high-concentration risk

A concentrated investor base amplifies redemption risk: one large exit
can force asset sales that affect all remaining investors. The largest-investor
scenario below connects MRS-31 and MRS-32 — it uses the investor register to
derive the fourth redemption stress scenario.

### 6.4 Counterparty Stress

AIFMD Annex VI requires AIFMs to stress-test **counterparty default** scenarios. For a long/short equity and credit strategy the key counterparties are the prime broker(s) and OTC derivatives (ISDA) counterparties. The relevant exposure is the **net uncollateralised** position after applying initial margin and variation margin held at the counterparty.

> Collateral coverage is simulated. A production system would derive these figures from the daily collateral reconciliation.


In [ ]:
# MRS-35: Counterparty stress — AIFM Hedge Fund
# Simulated prime brokerage and ISDA OTC derivatives counterparty register
# _cp_hf = rk.load_counterparty(FUND_ID) 
counterparty_stress = rk.compute_counterparty_stress(FUND_ID, ENGINE, NAV)
phtml.display_counterparty_stress(NAV, valuation_date=VALUATION_DATE, **counterparty_stress)


### 6.5 Combined Stress (Market + Liquidity)

ESMA/2020/1498 Annex VI requires a **combined stress** scenario: what happens if the market falls and investors simultaneously redeem? For this fund the combined shock is **equity −20%** (market stress) **and** **25% NAV redemption** (liquidity stress) applied simultaneously.

Under combined stress, liquid assets also decline in value — equity held as collateral or as liquidatable positions shrinks by 20%. The redemption demand is computed on the **pre-stress NAV** (investor expectation at point of submission).


In [ ]:
# MRS-35: Combined stress — equity -20% AND 25% redemption simultaneously
phtml.display_combined_stress_mkt_plus_liq(
    risk_df=risk_df,
    risk_df_liq=risk_df_liq,
    nav=NAV,
    notice_days=NOTICE,
    delta_equity=-0.20,
    redemption_pct=0.25,
    valuation_date=VALUATION_DATE,
);


#### 6.6. A note on reconciling exposures
<small>

| Measure | Numerator | Cash | Shorts | Derivatives |
|---|---|---|---|---|
| NAV | signed market value | included | netted | market value | 
| Liquidity buckets | abs(market value) | included | abs | abs(market value) | 
| Gross leverage | abs(market value) / notional | excluded | included | full notional | 
| Commitment leverage | net netting | excluded | netted | delta-adjusted notional | 

</small>

- **NAV** = sum of signed market values including cash. Shorts reduce NAV.
- **Liquidity bucket total** = sum of abs(market value) including cash. Always >= NAV due to short positions contributing positively.
- **Gross leverage numerator** = abs exposures excluding cash + derivative full notional. Excludes cash so will not match NAV.
- **Commitment leverage numerator** = uses delta-adjusted derivative notional and nets hedging positions. Excludes cash. As deltas are usually bellow 1, hedges are netted typically is less than gross leverage.

Conclusion: Gross and commitment are regulatory leverage metrics with specific inclusion/exclusion rules defined in EU 231/2013. They are not designed to reconcile to NAV; they measure economic risk exposure, not portfolio value.

## 7. P&L Attribution by Risk Factor

Under AIFMD Article 15, the risk function must be able to explain where
returns and losses come from, to answer "what drove the loss on day X" by factor.

This section uses **sensitivity-based attribution**. Regression-based
approaches give average historical loadings and cannot reflect current
position changes. Sensitivity-based attribution uses actual positions and
actual market moves each day, consistent with how VaR is computed.

> This is a risk governance output. It feeds the Board risk report and supports the AIFMD Article 15 evidence pack.

### Attribution framework

#### A. Total

$\text{Total P\&L} = \text{Equity P\&L} + \text{Rates P\&L} + \text{FX P\&L} + \text{Residual}$

---

#### B. Components

$\text{Equity P\&L} = \sum_i \beta_i \times r_{market} \times MV_i$

$\text{Rates P\&L} = \sum_i -D_i \times \Delta y \times MV_i$

$\text{FX P\&L} = \sum_i \text{notional}_{foreign,i} \times r_{FX,i}$

where $\beta_i$ is the 1-year rolling beta to the benchmark, $D_i$ is
modified duration, and $r_{FX,i}$ is the daily FX return vs EUR.

---

#### C. Residual

$\text{Residual} = \text{P\&L}_{actual} - (\text{Equity} + \text{Rates} + \text{FX})$

A large or persistent residual signals model limitations, missing factors
(credit spread, volatility, carry), or data issues. It is shown, not suppressed.

**% explained** $= |\text{explained}| / |\text{actual}|$. Target $\geq 80\%$.

In [ ]:
from src.risk.pnl_attribution import compute_daily_attribution
from src.ui.attribution_plot import plot_attribution_cumsum

# Compute attribution
attr_result = compute_daily_attribution(
    engine=ENGINE,
    fund_id=FUND_ID,
    bbg=BBG,
    valuation_date=VALUATION_DATE,
    residual_threshold_pct=0.20,
)

attr = attr_result['attr']
flagged = attr_result['flagged']
attr_cumsum = attr_result['attr_cumsum']

# Plot
plot_attribution_cumsum(attr_cumsum, FUND_ID, valuation_date=VALUATION_DATE)

# Model quality report
import src.print_utils as prt
prt.print_attribution(attr, flagged)


**Methodology and limitations**

Sensitivity-based attribution using actual daily positions and market moves.
Regression-based attribution is not used — it gives average historical loadings
and cannot reflect current position composition. 

Benchmark: SPY (S&P 500) — appropriate for this USD-heavy long/short equity portfolio.

**Limitations:**
* Single-factor equity model — no sector, style, or stock-level decomposition
* Rate attribution uses a simulated parallel shift — no key-rate DV01
* FX covers EUR/USD only
* Position composition is static in this simulation

**Regulatory context:** this section is an internal governance output consumed
by the Board and risk committee. It supports the AIFMD Article 15 evidence
pack and CSSF expectations for risk management reporting. It is not a direct
Annex IV or Annex VI deliverable.

---

## 8. Pre-Trade Compliance Check

The pre-trade check (`pre_trade_check` in `risk_utils.py`) models AIFMD Article 15(3)(c): the risk management function must be able to assess the impact of any proposed investment on the fund's risk profile **before** execution. It is run as a blocking gate — a trade flagging a breach is escalated to the PM and CRO before order placement.

`pre_trade_check` (in `risk_utils.py`) works in three steps:
1. Loads the current enriched portfolio via `get_risk_ready_df`
2. Constructs a pro-forma position set after applying the proposed trade (`_ptc_apply_trade`)
3. Runs AIFM Hedge Fund checks: gross leverage, commitment leverage, single-issuer \
concentration, sector concentration, counterparty exposure (OTC), short-selling \
reportability (EU 236/2012), and liquidity impact

### Checks performed

<small>

| # | Check | Limit | Basis |
|---|---|---|---|
| 1 | Gross leverage | 300% NAV | EU 231/2013 Art. 7; Recitals 13–14 |
| 2 | Commitment leverage | 200% NAV | EU 231/2013 Art. 8 |
| 3 | Single-issuer concentration | 25% NAV | Internal RMP |
| 4 | Sector concentration (GICS, corporate only) | 30% NAV | Internal RMP |
| 5 | Counterparty exposure (OTC derivatives) | 10% / 5% NAV | EU 231/2013 Art. 43 |
| 6 | Net short position | 0.2% NAV reportable | EU 236/2012 Art. 5 |
| 7 | Liquidity — days to liquidate | Fund-specific | AIFMD Art. 16 |

</small>


### Borrowings — Recital 13 correction

Borrowings are included in **both** gross and commitment exposure at absolute value (EU 231/2013 Recital 13). The only exclusion is capital call credit facilities that are temporary and fully covered by LP commitments (Recital 14) — not applicable to this hedge fund. The `pct_financed` field on each trade controls the funding source:

- `pct_financed = 0.0` — cash-funded: cash position reduced by the full notional, no new borrowing
- `pct_financed = 1.0` — prime broker financed: a `Borrowing` row is created in the pro-forma and counted in both leverage methods

### Pre-existing breaches

Checks [3] and [6] only flag a breach if the **proposed trade makes the breaches to get worse**. 


### 8.1  Scenario 1 — Clean buy (passes all checks)

New investment-grade bond, issuer not currently in the portfolio. \
EUR 3M notional on a fund NAV of approximately EUR 94.5M (≈ 3.2% of NAV). \
Post-trade gross leverage stays well below 300%; new issuer is below the 25% RMP concentration \
limit; eligible instrument. **Expected result: PASS.**

In [ ]:
from src.risk.leverage_computation import build_bbg_maps
counterparties_df = rk.load_counterparty(FUND_ID)
currency_bbg_map, deriv_bbg_map = build_bbg_maps(FUND_ID)

PTC_CTX = dict(
    engine=ENGINE,
    fund_id=FUND_ID,
    date=VALUATION_DATE,
    counterparties_df=counterparties_df,
    bbg=BBG,
    deriv_bbg_map=deriv_bbg_map,
    currency_bbg_map=currency_bbg_map,
)

In [ ]:
# Deutsche Bank IG bond — new name, small size
trade_pass = {
    'isin'           : 'XS2500000001',
    'direction'      : 'buy',
    'quantity'       : 3_000_000,
    'price_eur'      : 1.00,
    'counterparty'   : 'BNP Paribas',      
    'underlying_risk': 'Deutsche Bank AG',     
    'asset_class'    : 'Bond',
    'sub_asset_class': 'IG Corporate',
    'dur_adj_mid'    : 4.2,
    'currency'       : 'EUR',
    'adv_eur'        : 5_000_000,
    'issuer'         : 'Deutsche Bank AG',
    'sector'         : 'Financials',
    'pct_financed'   : 0.0,   # cash-funded IG bond buy

}

result_pass = rk.pre_trade_check(trade_pass, **PTC_CTX)
phtml.display_ptc(result_pass, test_number=1, valuation_date=VALUATION_DATE)
               





### 8.2  Scenario 2 — Gross and commitment leverage breach

A large new long equity position: 750,000 shares of Goldman Sachs at EUR 200/share \
= EUR 150M notional. The fund's current gross exposure is approximately EUR 152M on a \
NAV of EUR 94.5M (≈ 1.6×). Adding EUR 150M pushes gross exposure to ≈ EUR 302M = **3.20× NAV**, \
exceeding the 300% RMP limit. Commitment leverage also breaches. \
The single new issuer at EUR 150M would additionally represent ≈ 158% NAV, \
far above the 25% single-issuer concentration limit.

> This scenario deliberately stacks multiple breaches: it shows that `pre_trade_check` \
catches all of them simultaneously and presents each with its limit and actual value.

In [ ]:
# Goldman Sachs — oversized long, breaches gross leverage and issuer concentration
trade_leverage = {
    'isin'           : 'US38141G1040',
    'direction'      : 'buy',
    'quantity'       : 750_000,
    'price_eur'      : 200.00,
    'counterparty'   : 'Deutsche Bank AG',      
    'underlying_risk': 'Goldman Sachs',                 
    'sub_asset_class': 'Large Cap',
    'beta'           : 1.40,
    'currency'       : 'USD',
    'adv_eur'        : 150_000_000,
    'issuer'         : 'Goldman Sachs Group',
    'pct_financed'   : 1.0,   # fully prime-broker financed
}

result_pass = rk.pre_trade_check(trade_leverage, **PTC_CTX)
phtml.display_ptc(result_pass, test_number=2, valuation_date=VALUATION_DATE)


### 8.3  Scenario 3 — Single-issuer concentration breach only

 is not currently in the portfolio. Buying 150,000 shares at EUR 170/share \
= EUR 25.5M notional. On NAV of EUR 94.5M, this is **27.0%**, exceeding the 25% RMP \
single-issuer limit. The size is deliberately chosen to breach concentration while \
staying well within the 300% gross leverage limit (post-trade gross ≈ 1.88× NAV).

This is the typical operational question: the portfolio manager wants to open a meaningful \
new position. The risk function must confirm the proposed allocation is within limits before \
the order reaches the market. A minor reduction in size (below EUR 23.6M = 25% NAV) \
would resolve the breach.

In [ ]:
# Alphabet Inc — new position, sized to breach issuer concentration (27% NAV)
trade_conc = {
    'isin'           : 'US02079K3059',
    'direction'      : 'buy',
    'quantity'       : 150_000,           # 150 k × EUR 170 = EUR 25.5 M
    'price_eur'      : 170.00,
    'asset_class'    : 'Equity',
    'sub_asset_class': 'Large Cap',
    'beta'           : 1.15,
    'currency'       : 'USD',
    'adv_eur'        : 200_000_000,
    'issuer'         : 'Alphabet Inc',
    'sector'         : 'Technology',
    'pct_financed'   : 0.0,   # cash-funded equity buy
}

result_conc = rk.pre_trade_check(trade_conc, **PTC_CTX)
phtml.display_ptc(result_conc, test_number=3, valuation_date=VALUATION_DATE)


----


## Sustainability Risk Indicators

Sustainability risk monitoring is conducted as part of internal fund governance. The fund is SFDR Article 6 (no explicit sustainability objective).
Portfolio-level ESG scores are computed as NAV-weighted averages. 


Metrics monitored:
- Weighted average ESG score (composite, E, S, G)
- % of NAV in instruments with ESG score below 30*
- % of NAV with active controversy flag
- Weighted average carbon intensity (tCO2/EURm revenue)

> **Note**: ESG scores for liquid instruments are fetched from Bloomberg at
> enrichment time. Instruments without a Bloomberg ticker use fund administrator
> ESG data embedded in the position file.


> % of NAV in instruments with ESG score below internal threshold. 
> Threshold is defined in the Risk Management Policy. 
> ESG scores are not comparable across providers as each
> uses a different methodology and scale.
> (here using 30/100,Bloomberg scale 0-100 higher is better)

> **Scale note**: all ESG scores in this framework use a 0-100 scale where
> 100 is best, consistent with Bloomberg ESG methodology. Illiquid instrument
> scores are provided by the fund administrator on the same scale. In production,
> the ManCo should ensure all third-party ESG data is normalised to a consistent
> scale before aggregating portfolio-level metrics.

> **ESG look-through for derivatives**: derivatives have indirect ESG exposure
> through their underlying. The delta-adjusted notional is used as the ESG
> exposure weight rather than market value, which would understate the exposure.
>
> For options:
> $$ESG\_exposure_i = |\delta_i| \times Q_i \times \text{contract size} \times P_{underlying} \times FX$$
>
> For futures: full notional is used since delta = 1.
>
> For FX forwards: no ESG exposure assigned (no issuer).

In [ ]:
import src.risk.esg_utils as esg_u

esg_df  = esg_u.build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
esg_u.display_esg_assets(esg_df, valuation_date=VALUATION_DATE)


In [ ]:
esg_u.display_esg_summary(esg_df, valuation_date=VALUATION_DATE)


In [ ]:
esg_u.plot_esg_profile(esg_df, FUND_ID, plot_title='06. ESG profile - HF', valuation_date=VALUATION_DATE)


---

## 9. Annex IV Report 
<span style="color:gray"><i>MRS-34</i></span>

AIFMD Article 110 requires submission of a transparency report (Annex IV template) to the CSSF. Selected outputs from this risk snapshot can support Annex IV reporting. fund identity, strategy,
risk profile, leverage (gross and commitment method), and liquidity.

AIFMD II (Directive 2024/927/EU) expanded the leverage disclosure to a granular
breakdown by source: financial borrowing, synthetic leverage through derivatives,
and repo/reverse repo — making it easier for the CSSF to identify funds building
leverage through derivatives rather than direct borrowing.

**Regulatory basis:** AIFMD Art. 110 / EU 231/2013 Annex IV / ESMA v1.7 (Jul 2024)


In [ ]:
# Build Annex IV data
from src.ui.annex_iv_display import annex_iv_section
from src.reporting.annex_iv import build_annex_iv, export_annex_iv_excel
from src.ui.annex_iv_display import annex_iv_section

ANNEX_IV_QUARTER = '2026-03-31'
annex_iv = build_annex_iv(ENGINE, FUND_ID, quarter=ANNEX_IV_QUARTER)

# ═══════════════════════════════════════════════════════════════════════════════
# DISPLAY: All Annex IV sections
# ═══════════════════════════════════════════════════════════════════════════════

annex_iv_section(annex_iv, 'identification');
annex_iv_section(annex_iv, 'breakdown');
annex_iv_section(annex_iv, 'risk_measures');
annex_iv_section(annex_iv, 'leverage_detail');
annex_iv_section(annex_iv, 'liquidity_buckets');
annex_iv_section(annex_iv, 'liquidity_terms');

# Export to Excel
path = export_annex_iv_excel(ENGINE, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV workbook written: {path}")
